# Moshi Compression — Phase 2: Logit Alignment Training

**Goal.** Align the student's text logits with the teacher's distribution while
keeping the Phase-1 hidden alignment intact.

**Loss.**
- Primary (Phase-1 lock): hidden cosine loss, weight 1.0
- Secondary (NEW): sparse JSD on top-256 text logits, weight 0.1

**Teacher.** NOT loaded live. Uses the `topk_idx`/`topk_val` memmaps from the
Phase-0 cache — same files Phase 1 opened but ignored. A live teacher on
cuda:1 would take ~11.8 GB and leave insufficient headroom for activations.

**Gate to Phase 3:** validation text KL < 0.20 (mean over frames, full sparse-KL)

**Init:** `mhassann/moshi-p1-ckpt/ckpt_step_2058.pt` (cos_sim 0.896)

**Datasets required** (attach ALL before running):
- `mhassann/moshi-cache-s{0..2}p{0..3}` (12 datasets)
- `mhassann/moshi-cache-codes`
- `mhassann/moshi-p1-ckpt` (Phase-1 checkpoint)
- `tasfiatanha/moshi-frozen-heads`
- `tasfiatanha/moshi-repo`
- `tasfiatanha/moshi-compression-smoke`

**Session plan:** resume from Phase-1 checkpoint, train text JSD + hidden
cosine, ~6 000 steps/session @ ~2.5 s/step ≈ 4 h/session. Target: gate hit
in 1-3 sessions (text logits should converge faster than hidden did since
the backbone is already aligned).


## Cell 1 — Global patches

In [3]:
import os, sys
# MUST be set BEFORE `import torch` or the CUDA allocator ignores it.
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch._dynamo.config.disable = True
# sys.stdout.reconfigure(encoding="utf-8")
print("torch.compile disabled, expandable_segments enabled")
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))


torch.compile disabled, expandable_segments enabled
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


## Cell 2 — Environment verification

In [4]:
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

assert torch.cuda.device_count() >= 2, "Phase 2 needs dual GPU (student cuda:0, frozen heads cuda:1)"
torch.cuda.set_device(0)
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
=== environment check PASSED ===


## Cell 3 — Installs

In [5]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "einops",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes
print(f"moshi from: {moshi.__file__}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("=== installs OK ===")

import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
transformers: 4.44.2
bitsandbytes: 0.49.2
=== installs OK ===
CUDAGraphed monkey-patched to no-op


## Cell 4 — Write smol_temporal.py

Copy the SmolTemporalTransformer into the editable moshi install. The
autocast-wraps-adapters fix from Phase 1 is already baked into this file.


In [6]:
import pathlib, base64

DST = pathlib.Path('/kaggle/working/moshi_repo/moshi/models/smol_temporal.py')

# Same source as Phase-1 final: in_adapter + backbone + out_adapter all inside
# the fp16 autocast block. Trainables are fp16 in Phase 2 (no fp32 cast).
_SRC = '''# moshi/models/smol_temporal.py
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
import torch
import torch.nn as nn
import transformers
from ..modules.streaming import StreamingModule, State


@dataclass
class _SmolState(State):
    past_key_values: Optional[tuple] = field(default=None)

    def reset(self, reset_mask: torch.Tensor) -> None:
        super().reset(reset_mask)
        self.past_key_values = None


class SmolTemporalTransformer(StreamingModule[_SmolState]):
    def __init__(
        self,
        teacher_dim: int = 4096,
        student_dim: int = 2048,
        hf_name: str = "HuggingFaceTB/SmolLM2-1.7B",
        rope_theta: float = 10_000.0,
        device: str = "cuda:0",
        dtype: torch.dtype = torch.float16,
    ):
        super().__init__()
        self.teacher_dim = teacher_dim
        self.student_dim = student_dim

        cfg = transformers.AutoConfig.from_pretrained(hf_name)
        cfg.rope_theta = rope_theta
        cfg.use_cache = True
        cfg.attn_implementation = "eager"
        self.backbone = transformers.AutoModel.from_pretrained(
            hf_name, config=cfg, torch_dtype=dtype,
        )
        if hasattr(self.backbone, "embed_tokens"):
            self.backbone.embed_tokens = nn.Identity()

        self.in_adapter  = nn.Linear(teacher_dim, student_dim, bias=False)
        self.out_adapter = nn.Linear(student_dim, teacher_dim, bias=False)
        nn.init.normal_(self.in_adapter.weight,  std=1.0 / (teacher_dim ** 0.5))
        nn.init.normal_(self.out_adapter.weight, std=1.0 / (student_dim ** 0.5))

        self.to(device=device, dtype=dtype)

    def _init_streaming_state(self, batch_size: int) -> _SmolState:
        device = self.in_adapter.weight.device
        return _SmolState(batch_size=batch_size, device=device, past_key_values=None)

    def forward(
        self,
        x: torch.Tensor,
        cross_attention_src: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        assert cross_attention_src is None
        assert x.dim() == 3 and x.shape[-1] == self.teacher_dim

        x = x.to(self.in_adapter.weight.device)

        past_kv = (self._streaming_state.past_key_values
                   if self._streaming_state is not None else None)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            h = self.in_adapter(x)
            out = self.backbone(
                inputs_embeds=h,
                past_key_values=past_kv,
                use_cache=(self._streaming_state is not None),
                return_dict=True,
            )
            y = self.out_adapter(out.last_hidden_state)

        if self._streaming_state is not None:
            self._streaming_state.past_key_values = out.past_key_values

        return y

    def student_state_dict(self):
        return {
            "backbone":    self.backbone.state_dict(),
            "in_adapter":  self.in_adapter.state_dict(),
            "out_adapter": self.out_adapter.state_dict(),
        }

    def load_student_state_dict(self, sd: dict):
        self.backbone.load_state_dict(sd["backbone"])
        self.in_adapter.load_state_dict(sd["in_adapter"])
        self.out_adapter.load_state_dict(sd["out_adapter"])
'''

DST.write_text(_SRC)
print(f"Wrote {DST} ({DST.stat().st_size} bytes)")

import importlib, moshi.models
if hasattr(moshi.models, "smol_temporal"):
    importlib.reload(moshi.models.smol_temporal)
from moshi.models.smol_temporal import SmolTemporalTransformer
print(f"SmolTemporalTransformer imported OK: {SmolTemporalTransformer}")
print("=== Cell 4 PASSED ===")


Wrote /kaggle/working/moshi_repo/moshi/models/smol_temporal.py (3255 bytes)
SmolTemporalTransformer imported OK: <class 'moshi.models.smol_temporal.SmolTemporalTransformer'>
=== Cell 4 PASSED ===


## Cell 5 — Open cache memmap handles + codes

Phase 2 actually *uses* the `topk_idx`/`topk_val` memmaps as the teacher text
distribution. Same files Phase 1 opened but didn't train against.


In [7]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

WINDOWS_PER_PART = 5_000
TOTAL_WINDOWS    = 60_000
T_FRAMES         = 375
TEACHER_DIM      = 4096
N_CB             = 17
TOP_K            = 256
VAL_FRACTION     = 0.02
SEED             = 42

parts_hidden = []
parts_idx    = []
parts_val    = []

for shard in range(3):
    for part in range(4):
        for prefix in [
            f"/kaggle/input/moshi-cache-s{shard}p{part}",
            f"/kaggle/input/datasets/mhassann/moshi-cache-s{shard}p{part}",
        ]:
            import os
            if os.path.isdir(prefix):
                break
        parts_hidden.append(np.memmap(f"{prefix}/hidden.npy",
            dtype="float16", mode="r", shape=(5000, 375, 4096)))
        parts_idx.append(np.memmap(f"{prefix}/topk_idx.npy",
            dtype="int32",   mode="r", shape=(5000, 375, 256)))
        parts_val.append(np.memmap(f"{prefix}/topk_val.npy",
            dtype="float16", mode="r", shape=(5000, 375, 256)))

print(f"Opened {len(parts_hidden)} hidden + {len(parts_idx)} idx + {len(parts_val)} val memmap handles")

codes_path_candidates = [
    "/kaggle/input/moshi-cache-codes/codes.npy",
    "/kaggle/input/datasets/mhassann/moshi-cache-codes/codes.npy",
]
codes_path = None
for cp in codes_path_candidates:
    if os.path.exists(cp):
        codes_path = cp
        break
if codes_path is None:
    raise FileNotFoundError(f"codes.npy not found in: {codes_path_candidates}")

codes_mm = np.memmap(codes_path, dtype="int16", mode="r",
                     shape=(TOTAL_WINDOWS, N_CB, T_FRAMES))
print(f"Codes memmap: {codes_mm.shape} dtype={codes_mm.dtype}")

# Same seed + shuffle as Phase 1 → identical train/val split.
# Resuming a checkpoint that trained on these exact windows will not leak val.
rng = random.Random(SEED)
all_indices = list(range(TOTAL_WINDOWS))
rng.shuffle(all_indices)
n_val = max(1, int(TOTAL_WINDOWS * VAL_FRACTION))
val_indices = set(all_indices[:n_val])
train_indices = [i for i in range(TOTAL_WINDOWS) if i not in val_indices]
print(f"Train: {len(train_indices)}, Val: {n_val}")


class CacheDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        part_i, local_i = divmod(i, WINDOWS_PER_PART)

        h   = np.array(parts_hidden[part_i][local_i], copy=True)
        ti  = np.array(parts_idx[part_i][local_i], copy=True)
        tv  = np.array(parts_val[part_i][local_i], copy=True)
        c   = np.array(codes_mm[i], copy=True)

        return {
            "hidden":   torch.from_numpy(h),
            "topk_idx": torch.from_numpy(ti),
            "topk_val": torch.from_numpy(tv),
            "codes":    torch.from_numpy(c.astype(np.int64)),
        }


train_ds = CacheDataset(train_indices)
val_ds   = CacheDataset(list(val_indices))

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=0, pin_memory=True)

batch = train_ds[0]
print(f"Sample: hidden {batch['hidden'].shape} {batch['hidden'].dtype}, "
      f"codes {batch['codes'].shape} {batch['codes'].dtype}, "
      f"topk_val {batch['topk_val'].shape} {batch['topk_val'].dtype}")
print("=== Cell 5 PASSED ===")


Opened 12 hidden + 12 idx + 12 val memmap handles
Codes memmap: (60000, 17, 375) dtype=int16
Train: 58800, Val: 1200
Sample: hidden torch.Size([375, 4096]) torch.float16, codes torch.Size([17, 375]) torch.int64, topk_val torch.Size([375, 256]) torch.float16
=== Cell 5 PASSED ===


## Cell 6 — Build student model

Same split layout as Phase 1:
- cuda:0: student transformer (backbone + adapters, trainable fp16)
- cuda:1: frozen heads (emb, text_emb, out_norm, text_linear, depformer*)

Phase-1 checkpoint is loaded in Cell 8 (resume), NOT here. Here we just build
the shell and load the frozen head weights.


In [8]:
import torch, pathlib, gc
from moshi.models.loaders import CheckpointInfo
from moshi.models.smol_temporal import SmolTemporalTransformer
import pickle as _pickle

# numpy 2.x pickle compat (same as Phase 1)
class _NumpyCompatUnpickler(_pickle.Unpickler):
    _REMAP = {
        "numpy.core.multiarray": "numpy._core.multiarray",
        "numpy.core.numeric":    "numpy._core.numeric",
        "numpy.core.umath":      "numpy._core.umath",
        "numpy.core":            "numpy._core",
    }
    def find_class(self, module, name):
        return super().find_class(self._REMAP.get(module, module), name)

class _NpPickle:
    Unpickler        = _NumpyCompatUnpickler
    loads            = staticmethod(_pickle.loads)
    load             = staticmethod(_pickle.load)
    dump             = staticmethod(_pickle.dump)
    dumps            = staticmethod(_pickle.dumps)
    HIGHEST_PROTOCOL = _pickle.HIGHEST_PROTOCOL
    DEFAULT_PROTOCOL = _pickle.DEFAULT_PROTOCOL
    PickleError      = _pickle.PickleError
    UnpicklingError  = _pickle.UnpicklingError

def _torch_load(path, **kw):
    kw.setdefault("weights_only", False)
    kw.setdefault("pickle_module", _NpPickle)
    return torch.load(path, **kw)


def replace_temporal_transformer(lm_model, device="cpu", dtype=torch.float16):
    teacher_dim = getattr(lm_model, "transformer_dim", 4096)
    new_tt = SmolTemporalTransformer(
        teacher_dim=teacher_dim,
        student_dim=2048,
        hf_name="HuggingFaceTB/SmolLM2-1.7B",
        rope_theta=10_000.0,
        device=device,
        dtype=dtype,
    )
    old = lm_model.transformer
    lm_model.transformer = new_tt
    del old
    torch.cuda.empty_cache()
    for name, p in lm_model.named_parameters():
        if name.startswith("transformer."):
            p.requires_grad_(True)
        else:
            p.requires_grad_(False)
    return new_tt


from moshi.models.lm import LMModel

print("Building student shell from hardcoded moshiko config ...")
student_lm = LMModel(
    dim=4096, num_heads=32, num_layers=32, hidden_scale=4.125,
    gating="silu", norm="rms_norm_f32", positional_embedding="rope",
    context=3000, n_q=16, dep_q=8, card=2048, text_card=32000,
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    depformer_dim=1024, depformer_dim_feedforward=4224,
    depformer_num_heads=16, depformer_num_layers=6,
    depformer_multi_linear=True, depformer_weights_per_step=True,
    depformer_context=8, depformer_pos_emb="none",
    existing_text_padding_id=3,
).to(dtype=torch.float16)

print("Replacing Helium TT with SmolLM2-1.7B ...")
smol_tt = replace_temporal_transformer(student_lm, device="cpu", dtype=torch.float16)
gc.collect()
torch.cuda.empty_cache()

# Frozen heads
FROZEN_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-frozen-heads"),
    pathlib.Path("/kaggle/input/moshi-frozen-heads"),
]
frozen_dir = None
for p in FROZEN_CANDIDATES:
    if p.exists():
        frozen_dir = p
        break
if frozen_dir is None:
    raise FileNotFoundError(f"Frozen heads not found: {FROZEN_CANDIDATES}")

print(f"Loading frozen heads from {frozen_dir} ...")
frozen_modules = [
    "emb", "text_emb", "out_norm", "text_linear",
    "depformer_in", "depformer",
    "depformer_emb", "depformer_text_emb", "linears",
]
for name in frozen_modules:
    pt_file = frozen_dir / f"{name}.pt"
    if pt_file.exists():
        sd = torch.load(pt_file, map_location="cpu", weights_only=True)
        getattr(student_lm, name).load_state_dict(sd)

# Split GPU layout (same as Phase 1)
print("Moving student to split GPU layout ...")
student_lm.transformer.to("cuda:0")
for attr in ["emb", "text_emb", "out_norm", "text_linear", "depformer_in",
             "depformer", "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:1")

# Trainables stay fp16 — bnb optimizer holds fp32 master copy internally.
print("Keeping trainables in fp16 (bnb handles fp32 master)")

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} free {free/1e9:.2f} / {total/1e9:.2f} GB")

n_train = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
n_froz  = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
print(f"Trainable : {n_train/1e6:.1f} M  ({n_train/1e9:.3f} B)")
print(f"Frozen    : {n_froz/1e6:.1f} M  ({n_froz/1e9:.3f} B)")

# Cross-device forward_text patch (same as Phase 1)
import types as _types

def _fixed_forward_text(self, sequence, sum_condition=None, cross_attention_src=None):
    B, K, S = sequence.shape
    assert K == self.num_codebooks

    emb_device = next(self.emb[0].parameters()).device
    input_sequence = sequence.to(emb_device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    if sum_condition is not None:
        input_ = input_ + sum_condition.to(input_)
    if cross_attention_src is not None:
        cross_attention_src = cross_attention_src.to(input_)

    transformer_out = self.transformer(input_, cross_attention_src=cross_attention_src)

    if self.out_norm:
        on_device = next(self.out_norm.parameters()).device
        transformer_out = self.out_norm(transformer_out.to(on_device))

    tl_device = next(self.text_linear.parameters()).device
    text_logits = self.text_linear(transformer_out.to(tl_device))
    text_logits = text_logits[:, None]
    return transformer_out, text_logits

student_lm.forward_text = _types.MethodType(_fixed_forward_text, student_lm)
print("forward_text patched (cross-device)")
print("=== Cell 6 PASSED ===")


Building student shell from hardcoded moshiko config ...
Replacing Helium TT with SmolLM2-1.7B ...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading frozen heads from /kaggle/input/datasets/tasfiatanha/moshi-frozen-heads ...
Moving student to split GPU layout ...
Keeping trainables in fp16 (bnb handles fp32 master)
cuda:0 free 12.25 / 15.64 GB
cuda:1 free 13.30 / 15.64 GB
Trainable : 1627.5 M  (1.627 B)
Frozen    : 1110.8 M  (1.111 B)
forward_text patched (cross-device)
=== Cell 6 PASSED ===


## Cell 7 — Optimizer, scheduler, gradient checkpointing

Phase-2 LR is lower than Phase-1 default — we're fine-tuning an already-
converged backbone, not bootstrapping. Cosine decay from 5e-5 → 5e-6
across 20k steps (text alignment typically needs more steps than hidden did).


In [9]:
import bitsandbytes as bnb
import math

GRAD_ACCUM   = 4
LR           = 5e-5        # lower than Phase-1 (1e-4): fine-tune, not bootstrap
LR_MIN       = 5e-6
WARMUP_STEPS = 300         # short warmup, we're resuming from a converged state
MAX_STEPS    = 6_000       # per session; gate expected ~6k-20k total across sessions
MAX_NORM     = 5.0

trainable_params = [p for p in student_lm.parameters() if p.requires_grad]
assert len(trainable_params) > 0, "No trainable params!"

optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=LR, weight_decay=0.01)

# No-op GradScaler shim — same pattern that worked in Phase 1.
class _NoOpScaler:
    def scale(self, loss):   return loss
    def unscale_(self, opt): pass
    def step(self, opt):     opt.step()
    def update(self):        pass
    def get_scale(self):     return 1
    def state_dict(self):    return {}
    def load_state_dict(self, sd): pass
scaler = _NoOpScaler()

backbone = student_lm.transformer.backbone
backbone.config.use_cache = False
if hasattr(backbone, "gradient_checkpointing_enable"):
    backbone.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    gc_on = getattr(backbone, "is_gradient_checkpointing", False)
    print(f"Gradient checkpointing: is_gradient_checkpointing={gc_on}, "
          f"use_cache={backbone.config.use_cache}")
student_lm.train()
backbone.train()

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return max(LR_MIN / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

print(f"Optimizer: PagedAdamW8bit, lr={LR}, wd=0.01")
print(f"Scheduler: linear warmup {WARMUP_STEPS} steps, cosine -> {LR_MIN}")
print(f"Grad accum: {GRAD_ACCUM}, max_norm: {MAX_NORM}")
print(f"Max steps this session: {MAX_STEPS}")
print("=== Cell 7 PASSED ===")


Gradient checkpointing: is_gradient_checkpointing=True, use_cache=False
Optimizer: PagedAdamW8bit, lr=5e-05, wd=0.01
Scheduler: linear warmup 300 steps, cosine -> 5e-06
Grad accum: 4, max_norm: 5.0
Max steps this session: 6000
=== Cell 7 PASSED ===


## Cell 8 — Resume from Phase-1 checkpoint (or latest Phase-2 checkpoint)

Priority order:
1. `/kaggle/working/ckpt_step_*.pt`       (in-session Phase-2 checkpoints)
2. `/kaggle/input/.../moshi-p2-ckpt/`     (previous Phase-2 session)
3. `/kaggle/input/.../moshi-p1-ckpt/`     (fresh start from Phase-1 final)

When resuming from Phase-1: loads ONLY student weights, fresh optimizer state.
When resuming from Phase-2: loads weights + optimizer + scheduler + RNG.


In [10]:
import pathlib, glob, json, random
import numpy as np

# In-session / prior Phase-2 / Phase-1 init (fall through in this order)
P2_CANDIDATES = [
    "/kaggle/working",
    "/kaggle/input/datasets/mhassann/moshi-p2-ckpt",
    "/kaggle/input/moshi-p2-ckpt",
]
P1_CANDIDATES = [
    "/kaggle/input/datasets/mhassann/moshi-p1-ckpt",
    "/kaggle/input/moshi-p1-ckpt",
]

start_step = 0
total_wall = 0.0
ckpt_loaded = False
ckpt_phase  = None

# First try Phase-2 checkpoints (full resume)
for ckpt_dir in P2_CANDIDATES:
    if not os.path.isdir(ckpt_dir):
        continue
    ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
    if not ckpts:
        continue
    latest = ckpts[-1]
    print(f"Found Phase-2 checkpoint: {latest}")
    ckpt = _torch_load(latest, map_location="cpu")
    # Only consider it a P2 checkpoint if it was saved under phase="P2".
    # /kaggle/working can contain P1 ckpts from a previous Phase-1 run.
    if ckpt.get("phase") != "P2":
        print(f"  skipping — phase={ckpt.get('phase')} (not P2)")
        del ckpt
        continue

    smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
    smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
    smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
    print("  student weights restored")

    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print("  optimizer + scheduler + scaler restored")

    if "torch_rng" in ckpt:  torch.set_rng_state(ckpt["torch_rng"])
    if "cuda_rng" in ckpt:   torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    if "numpy_rng" in ckpt:  np.random.set_state(ckpt["numpy_rng"])
    if "python_rng" in ckpt: random.setstate(ckpt["python_rng"])

    start_step = ckpt.get("step", 0)
    total_wall = ckpt.get("wall_seconds", 0.0)
    ckpt_loaded = True
    ckpt_phase = "P2"
    del ckpt
    torch.cuda.empty_cache()
    print(f"  Resuming Phase-2 from step {start_step}")
    break

# Fallback: init from Phase-1 final (weights only, fresh optimizer)
if not ckpt_loaded:
    for ckpt_dir in P1_CANDIDATES:
        if not os.path.isdir(ckpt_dir):
            continue
        ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
        if not ckpts:
            continue
        latest = ckpts[-1]
        print(f"Initializing from Phase-1 checkpoint: {latest}")
        ckpt = _torch_load(latest, map_location="cpu")

        smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
        smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
        smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
        print("  Phase-1 student weights loaded (fresh optimizer state)")
        ckpt_loaded = True
        ckpt_phase = "P1-init"
        del ckpt
        torch.cuda.empty_cache()
        break

if not ckpt_loaded:
    raise RuntimeError(
        "No checkpoint found — Phase 2 requires either a Phase-1 final "
        "checkpoint or a prior Phase-2 checkpoint to resume from."
    )

print(f"Start step: {start_step}, phase init: {ckpt_phase}")
print("=== Cell 8 PASSED ===")


Initializing from Phase-1 checkpoint: /kaggle/input/datasets/mhassann/moshi-p1-ckpt/ckpt_step_2058.pt
  Phase-1 student weights loaded (fresh optimizer state)
Start step: 0, phase init: P1-init
=== Cell 8 PASSED ===


## Cell 9 — Loss functions

**Hidden cosine** (same as Phase 1, weight 1.0) — keeps the Phase-1 alignment
intact so text-logit training doesn't drift the backbone off-distribution.

**Sparse top-K JSD** (new, weight 0.1) — Jensen-Shannon divergence between
student logits and teacher top-256 distribution. We pull out the student
logits at the teacher's top-256 positions, renormalize both to that support,
then compute symmetric JSD = 0.5*(KL(p||m) + KL(q||m)) where m = (p+q)/2.

JSD is bounded in [0, ln2] which gives stable gradients even when student
and teacher disagree sharply. Full-vocab KL (32k positions) would also work
but with 255× more compute for the same signal.

The gate metric (val_text_kl) uses standard sparse KL, not JSD, to match
the execution doc's spec.


In [11]:
import torch
import torch.nn.functional as F

ALPHA_HIDDEN = 1.0
ALPHA_TEXT   = 0.1
VOCAB_SIZE   = 32000


def cosine_loss(student_h, teacher_h):
    s = student_h.float()
    t = teacher_h.float().to(s.device)
    cos = F.cosine_similarity(s, t, dim=-1)
    return (1.0 - cos).mean()


def sparse_text_jsd(student_logits, topk_idx, topk_val):
    """Symmetric JSD on the teacher's top-K support.

    student_logits: [B, T, V=32000] raw logits
    topk_idx:       [B, T, K=256]   int, teacher top-K positions
    topk_val:       [B, T, K=256]   teacher logits at those positions
    Returns: scalar JSD in [0, ln2], averaged over (B, T).

    Bounded in [0, ln(2)] ≈ 0.693. Gives stable gradients even when
    student and teacher disagree sharply.
    """
    d = student_logits.device
    idx = topk_idx.long().to(d)
    val = topk_val.float().to(d)

    # Student logits at teacher's top-K positions
    s_topk = torch.gather(student_logits.float(), 2, idx)  # [B, T, K]

    # Renormalize both to top-K support (softmax over K)
    p_student = F.softmax(s_topk, dim=-1)
    p_teacher = F.softmax(val,    dim=-1)
    log_p_s   = F.log_softmax(s_topk, dim=-1)
    log_p_t   = F.log_softmax(val,    dim=-1)

    m     = 0.5 * (p_student + p_teacher)
    log_m = m.clamp(min=1e-10).log()

    # Sum over K for each (B, T) position, then mean over (B, T)
    kl_pm = (p_student * (log_p_s - log_m)).sum(dim=-1).mean()
    kl_qm = (p_teacher * (log_p_t - log_m)).sum(dim=-1).mean()
    return 0.5 * (kl_pm + kl_qm)


def sparse_text_kl(student_logits, topk_idx, topk_val):
    """Plain sparse KL for the gate metric. Not used as training loss.

    student_logits: [B, T, V=32000]
    Returns: KL(teacher || student) over top-K support, averaged over (B, T).
    Unbounded above — can be large if student has not yet converged.
    """
    d = student_logits.device
    idx = topk_idx.long().to(d)
    val = topk_val.float().to(d)

    s_topk = torch.gather(student_logits.float(), 2, idx)
    log_p_student = F.log_softmax(s_topk, dim=-1)
    log_p_teacher = F.log_softmax(val,    dim=-1)
    p_teacher     = F.softmax(val, dim=-1)

    # KL(teacher || student) summed over K, mean over (B, T)
    kl = (p_teacher * (log_p_teacher - log_p_student)).sum(dim=-1)
    return kl.mean()


print(f"Loss functions defined: cosine_loss, sparse_text_jsd, sparse_text_kl")
print(f"ALPHA_HIDDEN={ALPHA_HIDDEN}, ALPHA_TEXT={ALPHA_TEXT}")
print("=== Cell 9 PASSED ===")


Loss functions defined: cosine_loss, sparse_text_jsd, sparse_text_kl
ALPHA_HIDDEN=1.0, ALPHA_TEXT=0.1
=== Cell 9 PASSED ===


## Cell 10 — Validation function

Reports both val_cos_sim (Phase-1 gate metric, should stay high) and
val_text_kl (Phase-2 gate metric, target < 0.20).


In [12]:
@torch.no_grad()
def validate(model, val_loader, device="cuda:0"):
    model.eval()
    total_1mc = 0.0
    total_kl  = 0.0
    n = 0
    for batch in val_loader:
        codes_b    = batch["codes"].to(device)
        hidden_t   = batch["hidden"].to("cuda:1")
        topk_idx_b = batch["topk_idx"].to("cuda:1")
        topk_val_b = batch["topk_val"].to("cuda:1")

        T_total = codes_b.shape[-1]
        T_CHUNK = 125
        n_chunks = (T_total + T_CHUNK - 1) // T_CHUNK
        chunk_1mc = 0.0
        chunk_kl  = 0.0
        for c_i in range(n_chunks):
            a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T_total)
            codes_c  = codes_b[..., a:b]
            hidden_c = hidden_t[:, a:b]
            idx_c    = topk_idx_b[:, a:b]
            val_c    = topk_val_b[:, a:b]

            transformer_out, text_logits = model.forward_text(codes_c)
            cos = F.cosine_similarity(transformer_out.float(),
                                      hidden_c.float().to(transformer_out.device),
                                      dim=-1)
            chunk_1mc += (1.0 - cos).mean().item() * (b - a) / T_total
            chunk_kl  += sparse_text_kl(text_logits[:, 0], idx_c, val_c).item() * (b - a) / T_total

        total_1mc += chunk_1mc
        total_kl  += chunk_kl
        n += 1

    model.train()
    return {
        "val_cos_sim":       1.0 - total_1mc / n,
        "val_1_minus_cos":   total_1mc / n,
        "val_text_kl":       total_kl  / n,
    }


print("validate() defined")
print("=== Cell 10 PASSED ===")


validate() defined
=== Cell 10 PASSED ===


## Cell 11 — Training loop

Same chunked structure as Phase-1 final. Per-chunk: forward → hidden cosine
loss + sparse JSD text loss → weighted sum → backward. Per-step (after
GRAD_ACCUM micro-steps): clip, optimizer step, log.

Checkpoint every 1000 steps, validate every 500 steps.
Gate: val_text_kl < 0.20 ends the phase.


In [13]:
import time, json, pathlib

OUT_DIR = pathlib.Path("/kaggle/working")
LOG_PATH = OUT_DIR / "train_log.jsonl"

student_lm.train()
device = "cuda:0"

log_file = open(LOG_PATH, "a")

step = start_step
micro_step = 0
accum_loss_h = 0.0
accum_loss_j = 0.0  # JSD (training loss component)
t_session = time.time()

print(f"Starting training from step {step}, max {MAX_STEPS} steps this session")
print(f"Grad accum = {GRAD_ACCUM}, effective batch size = {GRAD_ACCUM}")
print(f"Loss: {ALPHA_HIDDEN}*hidden_cos + {ALPHA_TEXT}*text_jsd")
print()

optimizer.zero_grad(set_to_none=True)

data_iter = iter(train_loader)

while step < start_step + MAX_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    codes_b    = batch["codes"].to(device)
    hidden_t   = batch["hidden"].to("cuda:1")
    topk_idx_b = batch["topk_idx"].to("cuda:1")
    topk_val_b = batch["topk_val"].to("cuda:1")

    T = codes_b.shape[-1]
    T_CHUNK = 125
    micro_h = 0.0
    micro_j = 0.0
    n_chunks = (T + T_CHUNK - 1) // T_CHUNK
    for c_i in range(n_chunks):
        a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T)
        codes_c  = codes_b[..., a:b]
        hidden_c = hidden_t[:, a:b]
        idx_c    = topk_idx_b[:, a:b]
        val_c    = topk_val_b[:, a:b]

        transformer_out, text_logits = student_lm.forward_text(codes_c)
        loss_h = cosine_loss(transformer_out, hidden_c)
        loss_j = sparse_text_jsd(text_logits[:, 0], idx_c, val_c)

        chunk_frac = (b - a) / T
        loss = (ALPHA_HIDDEN * loss_h + ALPHA_TEXT * loss_j) * chunk_frac / GRAD_ACCUM
        scaler.scale(loss).backward()
        micro_h += loss_h.item() * chunk_frac
        micro_j += loss_j.item() * chunk_frac

    accum_loss_h += micro_h
    accum_loss_j += micro_j
    micro_step += 1

    if micro_step % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        step += 1
        wall = time.time() - t_session + total_wall
        avg_h = accum_loss_h / GRAD_ACCUM
        avg_j = accum_loss_j / GRAD_ACCUM

        gn = float(grad_norm)
        gn_str = f"{gn:.3f}" if gn < 1e6 else "inf"

        row = {
            "step": step,
            "phase": "P2",
            "loss_hidden": round(avg_h, 5),
            "loss_text_jsd": round(avg_j, 5),
            "grad_norm": round(gn, 4) if gn < 1e6 else None,
            "lr": round(scheduler.get_last_lr()[0], 7),
            "wall_s": round(wall, 1),
        }
        log_file.write(json.dumps(row) + "\n")
        log_file.flush()

        if step % 50 == 0:
            print(f"step {step:5d}  hid={avg_h:.4f}  jsd={avg_j:.4f}  "
                  f"gn={gn_str}  lr={row['lr']:.2e}  wall={wall:.0f}s")

        accum_loss_h = 0.0
        accum_loss_j = 0.0

        if step % 500 == 0:
            val_result = validate(student_lm, val_loader, device)
            print(f"  VAL step {step}: cos_sim={val_result['val_cos_sim']:.4f}  "
                  f"text_kl={val_result['val_text_kl']:.4f}")
            val_row = {"step": step, "phase": "P2", "type": "val",
                       **val_result, "wall_s": round(wall, 1)}
            log_file.write(json.dumps(val_row) + "\n")
            log_file.flush()
            student_lm.train()

            if val_result["val_text_kl"] < 0.20:
                print(f"  *** GATE MET: val_text_kl = {val_result['val_text_kl']:.4f} < 0.20 ***")
                print("  Phase-2 complete! Ready for Phase-3.")

            # Also flag if hidden loss drifted (Phase-1 lock check)
            if val_result["val_1_minus_cos"] > 0.25:
                print(f"  ⚠ Hidden loss drifted: val_1_minus_cos = {val_result['val_1_minus_cos']:.4f} "
                      f"(Phase-1 baseline was 0.10). Consider raising ALPHA_HIDDEN or lowering ALPHA_TEXT.")

        if step % 1000 == 0:
            ckpt_path = OUT_DIR / f"ckpt_step_{step}.pt"
            ckpt = {
                "step": step,
                "phase": "P2",
                "wall_seconds": round(wall, 1),
                "student_backbone": smol_tt.backbone.state_dict(),
                "in_adapter": smol_tt.in_adapter.state_dict(),
                "out_adapter": smol_tt.out_adapter.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
                "numpy_rng": np.random.get_state(),
                "python_rng": random.getstate(),
                "torch_version": torch.__version__,
            }
            torch.save(ckpt, ckpt_path)
            size_gb = ckpt_path.stat().st_size / 1e9
            print(f"  CKPT saved: {ckpt_path.name} ({size_gb:.2f} GB)")
            del ckpt

log_file.close()

final_wall = time.time() - t_session + total_wall
print(f"\nTraining done. Final step: {step}, wall: {final_wall:.0f}s")

free, total_ = torch.cuda.mem_get_info(0)
print(f"cuda:0: free {free/1e9:.2f} / {total_/1e9:.2f} GB")

val_final = validate(student_lm, val_loader, device)
print(f"Final VAL: cos_sim={val_final['val_cos_sim']:.4f}  "
      f"text_kl={val_final['val_text_kl']:.4f}")
print("=== Cell 11 PASSED ===")


Starting training from step 0, max 6000 steps this session
Grad accum = 4, effective batch size = 4
Loss: 1.0*hidden_cos + 0.1*text_jsd

step    50  hid=0.1118  jsd=0.0408  gn=0.041  lr=8.30e-06  wall=119s
step   100  hid=0.1236  jsd=0.0428  gn=0.051  lr=1.67e-05  wall=235s
step   150  hid=0.1143  jsd=0.0422  gn=0.045  lr=2.50e-05  wall=351s
step   200  hid=0.1068  jsd=0.0340  gn=0.042  lr=3.33e-05  wall=468s
step   250  hid=0.0878  jsd=0.0281  gn=0.031  lr=4.17e-05  wall=585s
step   300  hid=0.1137  jsd=0.0423  gn=0.042  lr=5.00e-05  wall=703s
step   350  hid=0.0762  jsd=0.0299  gn=0.026  lr=5.00e-05  wall=820s
step   400  hid=0.0944  jsd=0.0340  gn=0.032  lr=5.00e-05  wall=938s
step   450  hid=0.1003  jsd=0.0403  gn=0.041  lr=4.99e-05  wall=1056s
step   500  hid=0.0974  jsd=0.0407  gn=0.036  lr=4.98e-05  wall=1173s
  VAL step 500: cos_sim=0.8998  text_kl=0.1509
  *** GATE MET: val_text_kl = 0.1509 < 0.20 ***
  Phase-2 complete! Ready for Phase-3.
step   550  hid=0.1084  jsd=0.0344  g

KeyboardInterrupt: 

## Cell 12 — Final checkpoint + push to Kaggle

Saves current-step checkpoint (if not already at a 1000-step boundary),
writes MANIFEST + dataset-metadata, pushes to `mhassann/moshi-p2-ckpt`.
Older checkpoints in /kaggle/working are deleted before upload.


In [14]:
import subprocess, json, pathlib, os

OUT_DIR = pathlib.Path("/kaggle/working")
username = os.environ.get("KAGGLE_USERNAME", "mhassann")

final_ckpt = OUT_DIR / f"ckpt_step_{step}.pt"
if not final_ckpt.exists():
    ckpt = {
        "step": step,
        "phase": "P2",
        "wall_seconds": round(time.time() - t_session + total_wall, 1),
        "student_backbone": smol_tt.backbone.state_dict(),
        "in_adapter": smol_tt.in_adapter.state_dict(),
        "out_adapter": smol_tt.out_adapter.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "torch_version": torch.__version__,
    }
    torch.save(ckpt, final_ckpt)
    print(f"Final ckpt: {final_ckpt.name} ({final_ckpt.stat().st_size/1e9:.2f} GB)")
    del ckpt

ckpts = sorted(OUT_DIR.glob("ckpt_step_*.pt"))
if len(ckpts) > 1:
    for old in ckpts[:-1]:
        old.unlink()
        print(f"  deleted old: {old.name}")

manifest = f"""# MANIFEST - moshi-p2-ckpt

Phase 2 logit alignment checkpoint.

| Key | Value |
|---|---|
| step | {step} |
| phase | P2 |
| gate_metric | val_text_kl |
| gate_target | < 0.20 |
| init_from | mhassann/moshi-p1-ckpt/ckpt_step_2058.pt |
| loss | 1.0*hidden_cos + 0.1*text_jsd (top-256 sparse) |
"""
(OUT_DIR / "MANIFEST.md").write_text(manifest)

dataset_id = f"{username}/moshi-p2-ckpt"
metadata = {
    "title":    "moshi-p2-ckpt",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

print("\nFiles to upload:")
upload_files = ["MANIFEST.md", "train_log.jsonl", "dataset-metadata.json"]
upload_files += [p.name for p in OUT_DIR.glob("ckpt_step_*.pt")]
total_gb = 0.0
for name in upload_files:
    p = OUT_DIR / name
    if p.exists():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {name:<35} {gb*1000:8.1f} MB")
print(f"  {'TOTAL':<35} {total_gb:8.2f} GB")

assert total_gb < 19, f"Upload too large: {total_gb:.1f} GB"

print("\nPushing dataset ...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")
if r.returncode == 0:
    print(f"SUCCESS (create) - kaggle.com/{dataset_id}")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump ...")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", f"P2 step {step}"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
    else:
        print(f"SUCCESS (version) - kaggle.com/{dataset_id}")

print("\n=== Phase-2 session COMPLETE ===")
print(f"Step: {step}, next session resumes from this checkpoint.")


Final ckpt: ckpt_step_611.pt (6.56 GB)

Files to upload:
  MANIFEST.md                              0.0 MB
  train_log.jsonl                          0.1 MB
  dataset-metadata.json                    0.0 MB
  ckpt_step_611.pt                      6561.8 MB
  TOTAL                                   6.56 GB

Pushing dataset ...
Skipping folder: .virtual_documents; use '--dir-mode' to upload folders
Starting upload for file train_log.jsonl
Upload successful: train_log.jsonl (79KB)
Starting upload for file ckpt_step_611.pt
Upload successful: ckpt_step_611.pt (6GB)
Starting upload for file MANIFEST.md
Upload successful: MANIFEST.md (291B)
Skipping folder: moshi_repo; use '--dir-mode' to upload folders
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mhassann/moshi-p2-ckpt

SUCCESS (create) - kaggle.com/mhassann/moshi-p2-ckpt

=== Phase-2 session COMPLETE ===
Step: 611, next session resumes from this checkpoint.
